In [1]:
from data.dataset import MalwareDatasetLoader, MalwareDataset
import pandas as pd
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, PowerTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from torch.utils.data import DataLoader

import torch
import torch.optim as optim
import multiprocessing
import concurrent

from neuralnets.model import DNN
from neuralnets.trainer import Trainer

loader = MalwareDatasetLoader()
df_train, df_val, df_test = loader.make_data_splits()

# local_orig, local_resp have only "-"" values
SKIPPED_COLUMNS = [
  'ts', 'uid', 'id.orig_h', 'id.resp_h', 'tunnel_parents', 'detailed-label', 'id.orig_p', 'id.resp_p', 'local_orig', 'local_resp']

ONE_HOT_COLUMNS = ['proto', 'service', 'conn_state', 'history']
NUMERIC_COLUMNS = [
   'duration', 'orig_bytes', 'resp_bytes', 'missed_bytes', 'orig_pkts', 'orig_ip_bytes', 'resp_pkts', 'resp_ip_bytes'
]
LABEL_COLUMN = 'label'

num_transformer = Pipeline(
  [
    ("imputer", SimpleImputer(missing_values=np.nan, strategy="constant", fill_value=-1)),
    ("power", PowerTransformer(method='yeo-johnson')),
    ("scalar", StandardScaler())
  ]
)
cat_transformer = Pipeline(
  [
    ("imputer", SimpleImputer(missing_values=np.nan, strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown='ignore', sparse_output=False))
  ]
)

preprocessor = ColumnTransformer(
    [("numeric", num_transformer, NUMERIC_COLUMNS),
    ("categorical", cat_transformer, ONE_HOT_COLUMNS)],
    remainder='passthrough'
)

def process_data(df, using_train_data):
    tmp_df = df[ONE_HOT_COLUMNS + NUMERIC_COLUMNS]
    # fit only on training data
    # only transforming for val and test data
    if using_train_data:
        X = preprocessor.fit_transform(tmp_df)
    else:
        X = preprocessor.transform(tmp_df)

    y = np.where(df[LABEL_COLUMN] == 'Benign', 1, 0)
    y = y.reshape(-1, 1)
    return X, y

X_train, y_train = process_data(df_train, using_train_data=True)
X_val, y_val = process_data(df_val, using_train_data=False)
X_test, y_test = process_data(df_test, using_train_data=False)

print(X_train.shape, y_train.shape)
print(X_val.shape, y_val.shape)
print(X_test.shape, y_test.shape)

IN_FEATURES = X_train.shape[1]
train_ds = MalwareDataset(X_train, y_train)
val_ds = MalwareDataset(X_val, y_val)
test_ds = MalwareDataset(X_test, y_test)

del loader 
del df_test, df_val, df_train
del X_train, y_train, X_val, y_val, X_test, y_test
import gc
gc.collect()



/data/src/stat841-project/STAT-841-cr-an-di/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using file: /home/cwswilki/.cache/kagglehub/datasets/agungpambudi/network-malware-detection-connection-analysis/versions/3/CTU-IoT-Malware-Capture-44-1conn.log.labeled.csv
Using file: /home/cwswilki/.cache/kagglehub/datasets/agungpambudi/network-malware-detection-connection-analysis/versions/3/CTU-IoT-Malware-Capture-9-1conn.log.labeled.csv
Using file: /home/cwswilki/.cache/kagglehub/datasets/agungpambudi/network-malware-detection-connection-analysis/versions/3/CTU-IoT-Malware-Capture-3-1conn.log.labeled.csv
Using file: /home/cwswilki/.cache/kagglehub/datasets/agungpambudi/network-malware-detection-connection-analysis/versions/3/CTU-IoT-Malware-Capture-34-1conn.log.labeled.csv
Using file: /home/cwswilki/.cache/kagglehub/datasets/agungpambudi/network-malware-detection-connection-analysis/versions/3/CTU-IoT-Malware-Capture-21-1conn.log.labeled.csv
Using file: /home/cwswilki/.cache/kagglehub/datasets/agungpambudi/network-malware-detection-connection-analysis/versions/3/CTU-IoT-Malware-Cap

/data/src/stat841-project/STAT-841-cr-an-di/data/dataset.py:28: DtypeWarning: Columns (8,9,10) have mixed types. Specify dtype option on import or set low_memory=False.
  dataframes.append(pd.read_csv(full_path, sep="|"))
/data/src/stat841-project/STAT-841-cr-an-di/data/dataset.py:31: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df = df.replace("-", np.nan)


Train: 17507702
Val: 3751650
Test: 3751651
(17507702, 263) (17507702, 1)
(3751650, 263) (3751650, 1)
(3751651, 263) (3751651, 1)


125

In [ ]:


from neuralnets.runner import training_run
import concurrent

if torch.cuda.is_available():
   multiprocessing.set_start_method('spawn', force=True)
   print("Multiprocessing start method set to 'spawn' for CUDA compatibility.")

PARALLEL=True
NUM_WORKERS = 2

RESET=False
if not RESET:
    try:
        print(f"Keeping: {len(accuracies)}")
    except Exception as e:
        accuracies = []
else:
    accuracies = []
futures = []
model_index = 0
SKIP_COUNT = 0
parameter_sets = []
for initial_lr in [0.001]:
    for weight_decay in [0.0001]:
        for dropout in [0.1]:
            for optim_type in ["adamw"]:
                for activation_type in ["sigmoid"]:
                    for batch_size in [1024]:
                        # NOTE: Better GPU memory efficiency, by having this iterable last, so all model sizes are on the GPU together.
                        for layers in [[512, 512, 512], [1024 for _ in range(20)] ]:
                            model_index += 1
                            if model_index <= SKIP_COUNT:
                                print(f"skipping: {model_index}")
                                continue
                            if optim_type == "sgd" and weight_decay != 0:
                                continue
                            parameter_sets.append((batch_size,
                                                    weight_decay, dropout, initial_lr,
                                                    optim_type, activation_type, layers))
import random
random.shuffle(parameter_sets)

with concurrent.futures.ProcessPoolExecutor(max_workers=NUM_WORKERS) as executor:
    for (batch_size, weight_decay, dropout, initial_lr, optim_type, activation_type, layers) in parameter_sets:
        desc = (f"Optimizer: {optim_type} - Weight Decay: {weight_decay} - Dropout: {dropout}"
            f" - initial LR: {initial_lr} - act: {activation_type} - layers: {layers} - batch_size:{batch_size}")
        print(f"SCHEDULING: {desc}")
        
        train_dataloader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, pin_memory=True)
        val_dataloader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)
        test_dataloader = DataLoader(test_ds, batch_size=batch_size, shuffle=False)
        if PARALLEL:
            futures.append(executor.submit(training_run,
                                        IN_FEATURES,
                                        batch_size,
                                        train_dataloader,
                                        val_dataloader,
                                        test_dataloader,
                                        weight_decay, dropout, initial_lr,
                            optim_type, activation_type, layers, log_progress=True,
                            ))
        else:
            result = training_run(IN_FEATURES,
                                batch_size,
                                        train_dataloader,
                                        val_dataloader,
                                        test_dataloader,
                                        weight_decay, dropout, initial_lr, optim_type,
                activation_type, layers)
            accuracies.append(result)

CATCH_EXCEPTIONS=False
if CATCH_EXCEPTIONS:
    for future in concurrent.futures.as_completed(futures):
        try:
            result = future.result()
            print(f"FINISHED RESULT: {result}")
            if result is not None:
                accuracies.append(result)
        except Exception as e:
            print(e)
else:
    for future in concurrent.futures.as_completed(futures):
        result = future.result()
        if result is not None:
            accuracies.append(result)



Multiprocessing start method set to 'spawn' for CUDA compatibility.
Keeping: 0
SCHEDULING: Optimizer: adamw - Weight Decay: 0.0001 - Dropout: 0.1 - initial LR: 0.001 - act: silu - layers: [1024, 1024, 1024, 1024, 1024, 1024, 1024, 1024, 1024, 1024] - batch_size:1024
SCHEDULING: Optimizer: adamw - Weight Decay: 0.0001 - Dropout: 0.1 - initial LR: 0.001 - act: silu - layers: [512, 512, 512] - batch_size:1024
Starting: Optimizer: adamw - Weight Decay: 0.0001 - Dropout: 0.1 - initial LR: 0.001 - act: silu - layers: [512, 512, 512] - batch_size:1024
Starting: Optimizer: adamw - Weight Decay: 0.0001 - Dropout: 0.1 - initial LR: 0.001 - act: silu - layers: [1024, 1024, 1024, 1024, 1024, 1024, 1024, 1024, 1024, 1024] - batch_size:1024
Using CUDA...
Using CUDA...


 91%|█████████ | 15484/17098 [01:50<00:08, 190.27it/s, loss=0.06]  

INFO | Optimizer: adamw - Weight Decay: 0.0001 - Dropout: 0.1 - initial LR: 0.001 - act: silu - layers: [512, 512, 512] - batch_size:1024
[EPOCH 1] LOSS : train=0.08250753587678969 val=1.5004574358137934 | ACCURACY : train=0.9751783184404749 val=0.8186906626911663


  0%|          | 0/17098 [00:00<?, ?it/s]

INFO | Optimizer: adamw - Weight Decay: 0.0001 - Dropout: 0.1 - initial LR: 0.001 - act: silu - layers: [1024, 1024, 1024, 1024, 1024, 1024, 1024, 1024, 1024, 1024] - batch_size:1024
[EPOCH 1] LOSS : train=0.07259700494971014 val=0.8815490779137507 | ACCURACY : train=0.9782599810794552 val=0.5640005991025523


...

INFO | Optimizer: adamw - Weight Decay: 0.0001 - Dropout: 0.1 - initial LR: 0.001 - act: silu - layers: [512, 512, 512] - batch_size:1024
[EPOCH 4] LOSS : train=0.06326869938508198 val=1.732249978880955 | ACCURACY : train=0.9815789657505594 val=0.8187301089930222


 31%|███       | 5222/17098 [00:33<01:03, 185.80it/s, loss=0.0626]

INFO | Test accuracy: 81.87301089930223 %
Figure(1200x500)
Final ACC: 0.8187301089930222


100%|██████████| 3664/3664 [00:09<00:00, 385.63it/s]


INFO | Optimizer: adamw - Weight Decay: 0.0001 - Dropout: 0.1 - initial LR: 0.001 - act: silu - layers: [1024, 1024, 1024, 1024, 1024, 1024, 1024, 1024, 1024, 1024] - batch_size:1024
[EPOCH 4] LOSS : train=0.062181749401387695 val=0.2563928567263497 | ACCURACY : train=0.9816821043024789 val=0.8915354192387068
INFO | Test accuracy: 89.15354192387068 %
Figure(1200x500)
Final ACC: 0.8915354192387068


In [ ]:
for (desc, val_acc) in accuracies:
    print(desc)
    print(val_acc)

 95%|█████████▌| 65311/68390 [02:09<00:04, 623.80it/s, loss=0.0867]

INFO | Optimizer: adamw - Weight Decay: 0 - Dropout: 0.2 - initial LR: 1 - act: relu - layers: [512] - batch_size:256
[EPOCH 1] LOSS : train=0.10061237853339632 val=0.09739512340472302 | ACCURACY : train=0.9716103563324054 val=0.9721155695116427


 21%|██        | 14454/68390 [00:20<01:05, 819.48it/s, loss=0.129] 

INFO | Optimizer: adamw - Weight Decay: 0 - Dropout: 0.2 - initial LR: 1 - act: relu - layers: [512, 512, 512] - batch_size:256
[EPOCH 1] LOSS : train=0.09825984652165451 val=0.10032705538729807 | ACCURACY : train=0.9716785069149233 val=0.9723002867583161


 78%|███████▊  | 53070/68390 [01:44<00:24, 621.56it/s, loss=0.082] 

...

 77%|███████▋  | 52605/68390 [01:29<00:21, 723.29it/s, loss=0.0907]

INFO | Optimizer: adamw - Weight Decay: 0 - Dropout: 0.2 - initial LR: 1 - act: relu - layers: [512, 512, 512] - batch_size:256
[EPOCH 4] LOSS : train=0.0969500678342088 val=0.10979686586744893 | ACCURACY : train=0.9720591170316483 val=0.9722984209275416


 31%|███       | 21347/68390 [00:41<01:17, 609.06it/s, loss=0.1]   

INFO | Optimizer: adamw - Weight Decay: 0 - Dropout: 0.2 - initial LR: 1 - act: relu - layers: [512] - batch_size:256
[EPOCH 5] LOSS : train=0.09676641543338758 val=0.09691797171648514 | ACCURACY : train=0.9720462850074807 val=0.9723221436331029


 13%|█▎        | 9226/68390 [00:16<01:41, 580.71it/s, loss=0.105]  